In [41]:
import sys
print(f"Python: {sys.version}")
print(f"Executável: {sys.executable}")

# Verifica dependências principais
try:
    import pyspark
    print(f"✓ PySpark: {pyspark.__version__}")
except:
    print("✗ PySpark não encontrado")

try:
    from track_platform import SSHTunnelManager
    print("✓ track_platform importado com sucesso")
except Exception as e:
    print(f"✗ Erro ao importar track_platform: {e}")

Python: 3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]
Executável: /Users/joseamaro/Documents/Projeto/data-pipeline-track/.venv/bin/python
✓ PySpark: 3.5.1
✓ track_platform importado com sucesso


In [42]:
import sys
print(f"Python: {sys.version}")
print(f"Executável: {sys.executable}")

Python: 3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]
Executável: /Users/joseamaro/Documents/Projeto/data-pipeline-track/.venv/bin/python


In [43]:
%run LoadEnvAndSetupSession.py

[INFO] Ambiente virtual detectado: /Users/joseamaro/Documents/Projeto/data-pipeline-track/.venv
Repo root: /Users/joseamaro/Documents/Projeto/data-pipeline-track
.env exists? True
[OK] pyspark encontrado: 3.5.1
track_platform: /Users/joseamaro/Documents/Projeto/data-pipeline-track/platform/shared-libs/track_platform/__init__.py
[OK] SparkSession criada com sucesso


In [44]:
import os

os.environ["SPARK_LOCAL_IP"] = "192.168.2.15"  # ajuste para o IP do seu Mac na rede

In [45]:
# ============================================================================
# VERIFICAR E GARANTIR QUE O SPARK ESTÁ FUNCIONANDO
# ============================================================================

# Verifica se o Spark foi criado corretamente pelo LoadEnvAndSetupSession
try:
    if 'spark' not in globals():
        print("[AVISO] Spark não encontrado em globals(), tentando recriar...")
        from track_platform.spark.session_manager import SparkSessionManager
        import os
        
        oracle_jdbc_coord = os.getenv("ORACLE_JDBC_COORD", "com.oracle.database.jdbc:ojdbc8:21.9.0.0")
        spark_extra_config = {
            "spark.jars.packages": oracle_jdbc_coord,
        }
        
        spark = SparkSessionManager.get_session(app_name="Oracle-Exploratorio", config=spark_extra_config)
        print("[OK] SparkSession recriada com sucesso")
    
    # Testa o Spark
    test_df = spark.range(1).limit(1)
    test_result = test_df.collect()
    
    print("=" * 70)
    print("✓ SPARK ESTÁ FUNCIONANDO")
    print("=" * 70)
    print(f"Versão: {spark.version}")
    print(f"App Name: {spark.sparkContext.appName}")
    print(f"Master: {spark.sparkContext.master}")
    
    try:
        ui_url = spark.sparkContext.uiWebUrl
        if ui_url:
            print(f"Spark UI: {ui_url}")
    except:
        pass
    
    print("=" * 70)
    
except NameError as e:
    if "spark" in str(e):
        print("=" * 70)
        print("✗ ERRO: Spark não está disponível")
        print("=" * 70)
        print("\n[SOLUÇÃO] Execute novamente a célula 3:")
        print("    %run LoadEnvAndSetupSession.py")
        print("\nOu reinicie o kernel do notebook.")
        print("=" * 70)
        raise
    else:
        raise
except Exception as e:
    print(f"[ERRO] Problema com Spark: {e}")
    print("[INFO] Tente reiniciar o kernel do notebook")
    raise


✓ SPARK ESTÁ FUNCIONANDO
Versão: 3.5.1
App Name: Oracle-Exploratorio
Master: local[*]
Spark UI: http://192.168.2.15:4041


In [46]:
# Configuração de túnel SSH para Oracle
import os
import sys
from pathlib import Path
import atexit

# Garante que .env está carregado (caso LoadEnvAndSetupSession não tenha carregado)
repo_root = Path.cwd()
while repo_root.name != "data-pipeline-track" and repo_root.parent != repo_root:
    repo_root = repo_root.parent

# Garante que o caminho está no sys.path
if str(repo_root / "platform" / "shared-libs") not in sys.path:
    sys.path.insert(0, str(repo_root / "platform" / "shared-libs"))

env_path = repo_root / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

# Tenta importar SSHTunnelManager
try:
    from track_platform import SSHTunnelManager
except ImportError:
    # Fallback: importação direta
    from track_platform.tunnel import SSHTunnelManager

# Cria túnel SSH se habilitado
ssh_tunnel_manager = SSHTunnelManager.from_env()
ssh_tunnel = None

# Ajusta remote_host se necessário
# Se o Oracle está no mesmo IP do bastion, usa o IP em vez de localhost
if ssh_tunnel_manager:
    oracle_host = os.getenv("ORACLE_SCOT_HOST") or os.getenv("ORACLE_GINF_HOST")
    if oracle_host and ssh_tunnel_manager.remote_host == "localhost":
        if oracle_host == ssh_tunnel_manager.ssh_host:
            print(f"[INFO] Oracle está no mesmo IP do bastion ({oracle_host})")
            print(f"[INFO] Ajustando remote_host de 'localhost' para '{oracle_host}'")
            ssh_tunnel_manager.remote_host = oracle_host
        else:
            print(f"[INFO] Oracle está em {oracle_host}, mas bastion é {ssh_tunnel_manager.ssh_host}")
            print(f"[INFO] Usando remote_host='localhost' (Oracle deve estar acessível via bastion)")

if ssh_tunnel_manager:
    print(f"[INFO] Túnel SSH configurado:")
    print(f"  - Bastion: {ssh_tunnel_manager.ssh_host}:{ssh_tunnel_manager.ssh_port}")
    print(f"  - Usuário: {ssh_tunnel_manager.ssh_user}")
    print(f"  - Host remoto: {ssh_tunnel_manager.remote_host}:{ssh_tunnel_manager.remote_port}")
    print(f"[INFO] Iniciando túnel SSH...")
    try:
        # Inicia o túnel diretamente (sem context manager para manter ativo)
        ssh_auth = {}
        if ssh_tunnel_manager.ssh_password:
            ssh_auth["ssh_password"] = ssh_tunnel_manager.ssh_password
            print(f"[INFO] Usando autenticação por senha")
        elif ssh_tunnel_manager.ssh_pkey:
            pkey_path = Path(ssh_tunnel_manager.ssh_pkey) if isinstance(ssh_tunnel_manager.ssh_pkey, str) else ssh_tunnel_manager.ssh_pkey
            if isinstance(pkey_path, Path) and pkey_path.exists():
                ssh_auth["ssh_pkey"] = str(pkey_path)
            else:
                ssh_auth["ssh_pkey"] = ssh_tunnel_manager.ssh_pkey
            print(f"[INFO] Usando autenticação por chave: {ssh_auth.get('ssh_pkey', 'N/A')}")
        
        from sshtunnel import SSHTunnelForwarder
        print(f"[INFO] Criando túnel: {ssh_tunnel_manager.ssh_host}:{ssh_tunnel_manager.ssh_port} -> {ssh_tunnel_manager.remote_host}:{ssh_tunnel_manager.remote_port}")
        ssh_tunnel_manager.tunnel = SSHTunnelForwarder(
            (ssh_tunnel_manager.ssh_host, ssh_tunnel_manager.ssh_port),
            ssh_username=ssh_tunnel_manager.ssh_user,
            remote_bind_address=(ssh_tunnel_manager.remote_host, ssh_tunnel_manager.remote_port),
            local_bind_address=("127.0.0.1", ssh_tunnel_manager.local_bind_port),
            set_keepalive=30,
            **ssh_auth,
        )
        print(f"[INFO] Iniciando túnel...")
        ssh_tunnel_manager.tunnel.start()
        ssh_tunnel_manager.local_bind_port = ssh_tunnel_manager.tunnel.local_bind_port
        ssh_tunnel = ssh_tunnel_manager
        
        print(f"[OK] Túnel SSH estabelecido!")
        print(f"  - Porta local: {ssh_tunnel.local_bind_port}")
        print(f"  - Status: {'ATIVO' if ssh_tunnel.is_active else 'INATIVO'}")
        print(f"  - Conexão Oracle: localhost:{ssh_tunnel.local_bind_port} -> {ssh_tunnel_manager.remote_host}:{ssh_tunnel_manager.remote_port}")
        
        # Registra função para encerrar túnel ao sair
        def cleanup_tunnel():
            if ssh_tunnel and ssh_tunnel.is_active:
                ssh_tunnel.tunnel.stop()
                print("[INFO] Túnel SSH encerrado")
        atexit.register(cleanup_tunnel)
    except Exception as e:
        print(f"[ERRO] Falha ao iniciar túnel SSH: {e}")
        import traceback
        traceback.print_exc()
        print("[WARN] Continuando sem túnel...")
        ssh_tunnel = None
        ssh_tunnel_manager = None
else:
    print("[INFO] Túnel SSH não configurado. Usando conexão direta.")
    ssh_tunnel = None

[INFO] Túnel SSH configurado:
  - Bastion: 10.255.150.11:22
  - Usuário: amaro.neto.beanalytic
  - Host remoto: 10.255.150.11:1521
[INFO] Iniciando túnel SSH...
[INFO] Usando autenticação por senha
[INFO] Criando túnel: 10.255.150.11:22 -> 10.255.150.11:1521
[INFO] Iniciando túnel...
[OK] Túnel SSH estabelecido!
  - Porta local: 53046
  - Status: ATIVO
  - Conexão Oracle: localhost:53046 -> 10.255.150.11:1521


In [47]:
# Diagnóstico do túnel SSH
if ssh_tunnel:
    print("=" * 60)
    print("DIAGNÓSTICO DO TÚNEL SSH")
    print("=" * 60)
    print(f"Túnel está ativo: {ssh_tunnel.is_active}")
    if ssh_tunnel.is_active:
        print(f"Porta local: {ssh_tunnel.local_bind_port}")
        print(f"Host remoto configurado: {ssh_tunnel.remote_host}:{ssh_tunnel.remote_port}")
        
        # Testa conectividade através do túnel
        import socket
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(5)
            result = sock.connect_ex(("127.0.0.1", ssh_tunnel.local_bind_port))
            sock.close()
            if result == 0:
                print(f"✓ Porta local {ssh_tunnel.local_bind_port} está acessível")
            else:
                print(f"✗ Porta local {ssh_tunnel.local_bind_port} não está acessível (código: {result})")
        except Exception as e:
            print(f"✗ Erro ao testar porta local: {e}")
    else:
        print("✗ Túnel não está ativo!")
    print("=" * 60)
else:
    print("[INFO] Túnel SSH não configurado para diagnóstico")

DIAGNÓSTICO DO TÚNEL SSH
Túnel está ativo: True
Porta local: 53046
Host remoto configurado: 10.255.150.11:1521
✓ Porta local 53046 está acessível


In [48]:
import os
from typing import Dict, Optional

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def _oracle_jdbc_config(prefix: str, tunnel=None, direct_config=None) -> Dict[str, str]:
    """
    Configura JDBC Oracle, usando túnel SSH se disponível.
    
    Args:
        prefix: Prefixo para variáveis de ambiente (SCOT, GINF, etc.)
        tunnel: Túnel SSH (opcional)
        direct_config: Dict com configuração direta {'host', 'port', 'service', 'user', 'password'} (opcional)
    """
    if direct_config:
        host = direct_config.get('host', '10.255.150.11')
        port = str(direct_config.get('port', '1521'))
        service = direct_config.get('service', 'bi.grupotracker.com.br')
        user = direct_config.get('user', 'C##AMARO_BE')
        password = direct_config.get('password', 'qiU!E0oe123')
        print(f"[INFO] Usando configuração direta: {host}:{port}/{service}")
    elif tunnel and tunnel.is_active:
        host = "127.0.0.1"
        port = str(tunnel.local_bind_port)
        service = os.getenv(f"ORACLE_{prefix}_SERVICE", "bi.grupotracker.com.br")
        user = os.getenv(f"ORACLE_{prefix}_USER", "C##AMARO_BE")
        password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "qiU!E0oe123")
        print(f"[INFO] Usando túnel SSH: {host}:{port} (redirecionando para {tunnel.remote_host}:{tunnel.remote_port})")
    else:
        host = os.getenv(f"ORACLE_{prefix}_HOST", "10.255.150.11")
        port = os.getenv(f"ORACLE_{prefix}_PORT", "50412")
        service = os.getenv(f"ORACLE_{prefix}_SERVICE", "bi.grupotracker.com.br")
        user = os.getenv(f"ORACLE_{prefix}_USER", "C##AMARO_BE")
        password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "qiU!E0oe123")
        print(f"[INFO] Usando conexão direta: {host}:{port}")
    
    jdbc_url = f"jdbc:oracle:thin:@//{host}:{port}/{service}"
    return {
        "url": jdbc_url,
        "properties": {
            "user": user,
            "password": password,
            "driver": "oracle.jdbc.driver.OracleDriver",
        },
    }


# ============================================================================
# CONFIGURAÇÃO ORACLE - CREDENCIAIS DIRETAS
# ============================================================================
# Configuração fornecida pelo usuário:
# URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br
# User: C##AMARO_BE
# Senha: qiU!E0oe123
# Schema: GINF

direct_oracle_config = {
    'host': '10.255.150.11',
    'port': '1521',
    'service': 'bi.grupotracker.com.br',
    'user': 'C##AMARO_BE',
    'password': 'qiU!E0oe123'
}

# Configurações para SCOT e GINF usando as mesmas credenciais
oracle_scot_cfg = _oracle_jdbc_config("SCOT", tunnel=ssh_tunnel if 'ssh_tunnel' in globals() else None, direct_config=direct_oracle_config)
oracle_ginf_cfg = _oracle_jdbc_config("GINF", tunnel=ssh_tunnel if 'ssh_tunnel' in globals() else None, direct_config=direct_oracle_config)

print(f"\n{'=' * 70}")
print("CONFIGURAÇÃO ORACLE")
print("=" * 70)
print(f"URL SCOT: {oracle_scot_cfg['url']}")
print(f"URL GINF: {oracle_ginf_cfg['url']}")
print(f"User: {direct_oracle_config['user']}")
print(f"Service: {direct_oracle_config['service']}")
print("=" * 70)

oracle_ginf_cfg


[INFO] Usando configuração direta: 10.255.150.11:1521/bi.grupotracker.com.br
[INFO] Usando configuração direta: 10.255.150.11:1521/bi.grupotracker.com.br

CONFIGURAÇÃO ORACLE
URL SCOT: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br
URL GINF: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br
User: C##AMARO_BE
Service: bi.grupotracker.com.br


{'url': 'jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br',
 'properties': {'user': 'C##AMARO_BE',
  'password': 'qiU!E0oe123',
  'driver': 'oracle.jdbc.driver.OracleDriver'}}

In [49]:
# ============================================================================
# TESTAR SERVIÇOS ORACLE ENCONTRADOS NO LISTENER
# ============================================================================

# Serviços encontrados via lsnrctl services:
oracle_services_found = [
    "bi.grupotracker.com.br",
    "dev.grupotracker.com.br",
    "hml.grupotracker.com.br",
    "CDBQA_grupotracker.grupotracker.com.br",
    "CDBQA_DEV.paas.oracle.com",
    "CDBQA_HML.paas.oracle.com",
    "CDBQAXDB.grupotracker.com.br",
    "339cad48f799751be0630b96ff0aa8d2.grupotracker.com.br",
    "311d46f4e18f7549e0630b96ff0acc64.grupotracker.com.br",
]

def test_oracle_services(services_list, tunnel=None):
    """Testa uma lista de service names Oracle e retorna o que funciona."""
    import socket
    from typing import Optional, Dict, Any
    
    print("=" * 70)
    print("TESTANDO SERVIÇOS ORACLE")
    print("=" * 70)
    
    working_service = None
    
    for service in services_list:
        print(f"\n[TESTE] Tentando service: {service}")
        
        try:
            if tunnel and tunnel.is_active:
                host = "127.0.0.1"
                port = tunnel.local_bind_port
            else:
                host = os.getenv("ORACLE_SCOT_HOST", "localhost")
                port = int(os.getenv("ORACLE_SCOT_PORT", "50412"))
            
            # Testa conectividade na porta
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(2)
            result = sock.connect_ex((host, port))
            sock.close()
            
            if result != 0:
                print(f"    ✗ Porta {port} não acessível (código: {result})")
                continue
            
            # Tenta criar configuração JDBC
            jdbc_url = f"jdbc:oracle:thin:@//{host}:{port}/{service}"
            test_cfg = {
                "url": jdbc_url,
                "properties": {
                    "user": os.getenv("ORACLE_SCOT_USER", "C##AMARO_BE"),
                    "password": os.getenv("ORACLE_SCOT_PASSWORD", "qiU!E0oe123"),
                    "driver": "oracle.jdbc.driver.OracleDriver"
                }
            }
            
            # Tenta uma query simples
            try:
                test_df = spark.read.jdbc(
                    test_cfg["url"],
                    table="(SELECT 1 as test FROM dual) test_alias",
                    properties=test_cfg["properties"]
                )
                test_df.collect()
                print(f"    ✓ Service '{service}' FUNCIONA!")
                working_service = service
                print(f"\n{'=' * 70}")
                print(f"SERVIÇO FUNCIONAL ENCONTRADO: {service}")
                print(f"{'=' * 70}")
                print(f"\nConfigure no .env:")
                print(f"  ORACLE_SCOT_SERVICE={service}")
                print(f"\nOu use diretamente:")
                print(f"  oracle_scot_cfg = _oracle_jdbc_config('SCOT', tunnel=ssh_tunnel)")
                break
            except Exception as e:
                error_msg = str(e)
                if "ORA-12514" in error_msg:
                    print(f"    ✗ Service não reconhecido pelo listener")
                elif "ORA-01017" in error_msg:
                    print(f"    ⚠ Service reconhecido, mas credenciais inválidas (isso é bom!)")
                    working_service = service
                    print(f"\n{'=' * 70}")
                    print(f"SERVIÇO RECONHECIDO (credenciais precisam ser ajustadas): {service}")
                    print(f"{'=' * 70}")
                    break
                else:
                    print(f"    ✗ Erro: {error_msg[:100]}")
        
        except Exception as e:
            print(f"    ✗ Erro ao testar: {e}")
    
    if not working_service:
        print(f"\n{'=' * 70}")
        print("NENHUM SERVIÇO FUNCIONOU")
        print("=" * 70)
        print("\nVerifique:")
        print("  1. Credenciais Oracle no .env")
        print("  2. Túnel SSH está ativo")
        print("  3. Service names corretos")
    
    return working_service


# Executa teste
print("\n")
working_service = test_oracle_services(oracle_services_found, tunnel=ssh_tunnel if 'ssh_tunnel' in globals() else None)




TESTANDO SERVIÇOS ORACLE

[TESTE] Tentando service: bi.grupotracker.com.br
    ⚠ Service reconhecido, mas credenciais inválidas (isso é bom!)

SERVIÇO RECONHECIDO (credenciais precisam ser ajustadas): bi.grupotracker.com.br


In [57]:
# ============================================================================
# TESTE RÁPIDO DE CONEXÃO ORACLE
# ============================================================================

def test_oracle_connection(cfg):
    """Testa conexão Oracle com a configuração fornecida."""
    try:
        print("=" * 70)
        print("TESTANDO CONEXÃO ORACLE")
        print("=" * 70)
        print(f"URL: {cfg['url']}")
        print(f"User: {cfg['properties']['user']}")
        print(f"Password: {'*' * len(cfg['properties']['password'])}")
        print("\n[INFO] Tentando conectar...")
        
        test_df = spark.read.jdbc(
            cfg["url"],
            table="(SELECT 'OK' as status, SYSDATE as current_time FROM dual) test_alias",
            properties=cfg["properties"]
        )
        result = test_df.collect()
        
        print("✓ CONEXÃO BEM-SUCEDIDA!")
        print(f"  Status: {result[0]['status']}")
        print(f"  Hora do servidor: {result[0]['current_time']}")
        return True
        
    except Exception as e:
        error_msg = str(e)
        print("✗ FALHA NA CONEXÃO")
        
        if "ORA-01017" in error_msg:
            print("\n[ERRO] Credenciais inválidas (ORA-01017)")
            print("  Verifique ORACLE_SCOT_USER e ORACLE_SCOT_PASSWORD no .env")
        elif "ORA-12514" in error_msg:
            print("\n[ERRO] Service name não reconhecido (ORA-12514)")
            print("  Verifique ORACLE_SCOT_SERVICE no .env")
        elif "Connection refused" in error_msg or "Network is unreachable" in error_msg:
            print("\n[ERRO] Problema de conectividade")
            print("  Verifique se o túnel SSH está ativo")
        else:
            print(f"\n[ERRO] {error_msg[:200]}")
        
        return False


# Testa conexão com configuração atual
if 'oracle_scot_cfg' in globals():
    test_oracle_connection(oracle_scot_cfg)
else:
    print("[AVISO] Execute primeiro a célula de configuração Oracle (célula 7)")


TESTANDO CONEXÃO ORACLE
✗ FALHA NA CONEXÃO

[ERRO] 'NoneType' object is not subscriptable


2025-12-15 06:32:38,010| ERROR   | Socket exception: Can't assign requested address (49)
2025-12-15 06:32:40,321| ERROR   | Socket exception: Can't assign requested address (49)


In [51]:
# ============================================================================
# VERIFICAR CREDENCIAIS ORACLE CONFIGURADAS
# ============================================================================

def show_oracle_credentials(prefix="SCOT", show_password=False):
    """Mostra as credenciais Oracle configuradas de forma segura."""
    import os
    
    print("=" * 70)
    print(f"CREDENCIAIS ORACLE - {prefix}")
    print("=" * 70)
    
    # Variáveis de ambiente
    service = os.getenv(f"ORACLE_{prefix}_SERVICE", "NÃO CONFIGURADO")
    user = os.getenv(f"ORACLE_{prefix}_USER", "NÃO CONFIGURADO")
    password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "NÃO CONFIGURADO")
    host = os.getenv(f"ORACLE_{prefix}_HOST", "NÃO CONFIGURADO")
    port = os.getenv(f"ORACLE_{prefix}_PORT", "NÃO CONFIGURADO")
    
    print(f"\n[CONFIGURAÇÃO]")
    print(f"  Service Name: {service}")
    print(f"  Host: {host}")
    print(f"  Port: {port}")
    print(f"  User: {user}")
    
    if show_password:
        print(f"  Password: {password}")
    else:
        if password != "NÃO CONFIGURADO" and password:
            masked = "*" * min(len(password), 20)
            print(f"  Password: {masked} (oculto)")
            print(f"  [INFO] Para ver a senha, execute: show_oracle_credentials('{prefix}', show_password=True)")
        else:
            print(f"  Password: {password}")
    
    # Verifica se há configuração no objeto oracle_scot_cfg
    if f'oracle_{prefix.lower()}_cfg' in globals():
        cfg = globals()[f'oracle_{prefix.lower()}_cfg']
        print(f"\n[CONFIGURAÇÃO ATIVA (do objeto)]")
        print(f"  URL: {cfg.get('url', 'N/A')}")
        print(f"  User: {cfg.get('properties', {}).get('user', 'N/A')}")
        if show_password:
            print(f"  Password: {cfg.get('properties', {}).get('password', 'N/A')}")
        else:
            pwd = cfg.get('properties', {}).get('password', '')
            if pwd:
                masked = "*" * min(len(pwd), 20)
                print(f"  Password: {masked} (oculto)")
    
    # Verifica arquivo .env
    print(f"\n[ARQUIVO .env]")
    env_path = ".env"
    try:
        with open(env_path, 'r') as f:
            env_lines = f.readlines()
        
        oracle_vars = {}
        for line in env_lines:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                key = key.strip()
                value = value.split('#')[0].strip().strip('"').strip("'")
                if key.startswith(f"ORACLE_{prefix}_"):
                    oracle_vars[key] = value
        
        if oracle_vars:
            for key, value in sorted(oracle_vars.items()):
                if 'PASSWORD' in key and not show_password:
                    masked = "*" * min(len(value), 20) if value else "NÃO CONFIGURADO"
                    print(f"  {key}: {masked} (oculto)")
                else:
                    print(f"  {key}: {value}")
        else:
            print(f"  Nenhuma variável ORACLE_{prefix}_* encontrada no .env")
    except FileNotFoundError:
        print(f"  Arquivo .env não encontrado")
    except Exception as e:
        print(f"  Erro ao ler .env: {e}")
    
    print("=" * 70)
    
    return {
        'service': service,
        'user': user,
        'password': password if show_password else None,
        'host': host,
        'port': port
    }


# Mostra credenciais (senha oculta por padrão)
print("\n")
creds = show_oracle_credentials("SCOT", show_password=False)

# Para ver a senha, descomente a linha abaixo:
# creds = show_oracle_credentials("SCOT", show_password=True)




CREDENCIAIS ORACLE - SCOT

[CONFIGURAÇÃO]
  Service Name: bi.grupotracker.com.br
  Host: 10.255.150.11
  Port: 1521
  User: C##AMARO_BE
  Password: *********** (oculto)
  [INFO] Para ver a senha, execute: show_oracle_credentials('SCOT', show_password=True)

[CONFIGURAÇÃO ATIVA (do objeto)]
  URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br
  User: C##AMARO_BE
  Password: *********** (oculto)

[ARQUIVO .env]
  Arquivo .env não encontrado


In [52]:
# ============================================================================
# CONSULTAR USUÁRIOS E PERMISSÕES NO ORACLE
# ============================================================================

def query_oracle_users(cfg, current_user_only=False):
    """
    Consulta usuários Oracle e suas permissões.
    
    Args:
        cfg: Configuração Oracle (oracle_scot_cfg)
        current_user_only: Se True, mostra apenas o usuário atual
    """
    try:
        print("=" * 70)
        print("CONSULTANDO USUÁRIOS ORACLE")
        print("=" * 70)
        
        if current_user_only:
            query = """
                SELECT 
                    USER as username,
                    SYS_CONTEXT('USERENV', 'SESSION_USER') as session_user,
                    SYS_CONTEXT('USERENV', 'CURRENT_USER') as current_user,
                    SYS_CONTEXT('USERENV', 'AUTHENTICATED_IDENTITY') as authenticated_identity,
                    SYS_CONTEXT('USERENV', 'AUTHENTICATION_METHOD') as auth_method
                FROM dual
            """
            print("\n[1] Usuário atual da sessão:")
        else:
            query = """
                SELECT 
                    username,
                    account_status,
                    created,
                    default_tablespace,
                    temporary_tablespace,
                    profile
                FROM all_users
                WHERE username = USER
                ORDER BY username
            """
            print("\n[1] Informações do usuário atual:")
        
        df = spark.read.jdbc(
            cfg["url"],
            table=f"({query}) user_info",
            properties=cfg["properties"]
        )
        
        df.show(truncate=False)
        
        print("\n[2] Schemas acessíveis pelo usuário atual:")
        query2 = """
            SELECT DISTINCT
                owner as schema_name,
                COUNT(DISTINCT table_name) as table_count
            FROM all_tables
            WHERE owner NOT IN ('SYS', 'SYSTEM', 'SYSAUX')
            GROUP BY owner
            ORDER BY owner
        """
        
        df2 = spark.read.jdbc(
            cfg["url"],
            table=f"({query2}) schemas",
            properties=cfg["properties"]
        )
        df2.show(truncate=False)
        
        print("\n[3] Permissões de sistema do usuário atual:")
        query3 = """
            SELECT 
                privilege,
                admin_option
            FROM user_sys_privs
            ORDER BY privilege
        """
        
        try:
            df3 = spark.read.jdbc(
                cfg["url"],
                table=f"({query3}) sys_privs",
                properties=cfg["properties"]
            )
            df3.show(truncate=False)
        except Exception as e:
            print(f"  [INFO] Não foi possível consultar permissões: {e}")
        
        print("\n[4] Roles do usuário atual:")
        query4 = """
            SELECT 
                granted_role,
                admin_option,
                default_role
            FROM user_role_privs
            ORDER BY granted_role
        """
        
        try:
            df4 = spark.read.jdbc(
                cfg["url"],
                table=f"({query4}) roles",
                properties=cfg["properties"]
            )
            df4.show(truncate=False)
        except Exception as e:
            print(f"  [INFO] Não foi possível consultar roles: {e}")
        
        print("=" * 70)
        
    except Exception as e:
        error_msg = str(e)
        print("✗ ERRO ao consultar usuários")
        
        if "ORA-01017" in error_msg:
            print("\n[ERRO] Credenciais inválidas")
            print("  As credenciais atuais não permitem acesso ao Oracle")
        elif "ORA-12514" in error_msg:
            print("\n[ERRO] Service name não reconhecido")
        else:
            print(f"\n[ERRO] {error_msg[:200]}")


# Comandos SQL que você pode executar diretamente no servidor Oracle via SQL*Plus ou SQLcl
print("""
======================================================================
COMANDOS PARA EXECUTAR DIRETAMENTE NO SERVIDOR ORACLE
======================================================================

1. Conectar ao Oracle via SSH:
   ssh amaro.neto.beanalytic@10.255.150.11
   sqlplus / as sysdba
   # ou
   sqlplus usuario/senha@dev.grupotracker.com.br

2. Ver usuário atual:
   SELECT USER FROM dual;
   SELECT SYS_CONTEXT('USERENV', 'SESSION_USER') FROM dual;

3. Ver todos os usuários (como DBA):
   SELECT username, account_status, created, default_tablespace 
   FROM dba_users 
   ORDER BY username;

4. Ver usuários comuns (não-DBA):
   SELECT username, account_status, created 
   FROM all_users 
   WHERE username = USER;

5. Ver schemas/tablespaces acessíveis:
   SELECT DISTINCT owner 
   FROM all_tables 
   ORDER BY owner;

6. Ver permissões do usuário atual:
   SELECT * FROM user_sys_privs;
   SELECT * FROM user_role_privs;
   SELECT * FROM user_tab_privs;

7. Ver service names registrados:
   lsnrctl services
   # ou
   SELECT name, value FROM v$parameter WHERE name = 'service_names';

8. Ver conexões ativas:
   SELECT username, machine, program, status 
   FROM v$session 
   WHERE username IS NOT NULL;

======================================================================
""")

# Se tiver configuração Oracle, tenta consultar
if 'oracle_scot_cfg' in globals():
    print("\n[INFO] Tentando consultar informações do usuário atual...\n")
    query_oracle_users(oracle_scot_cfg, current_user_only=True)
else:
    print("\n[AVISO] Execute primeiro a célula de configuração Oracle (célula 7)")



COMANDOS PARA EXECUTAR DIRETAMENTE NO SERVIDOR ORACLE

1. Conectar ao Oracle via SSH:
   ssh amaro.neto.beanalytic@10.255.150.11
   sqlplus / as sysdba
   # ou
   sqlplus usuario/senha@dev.grupotracker.com.br

2. Ver usuário atual:
   SELECT USER FROM dual;
   SELECT SYS_CONTEXT('USERENV', 'SESSION_USER') FROM dual;

3. Ver todos os usuários (como DBA):
   SELECT username, account_status, created, default_tablespace 
   FROM dba_users 
   ORDER BY username;

4. Ver usuários comuns (não-DBA):
   SELECT username, account_status, created 
   FROM all_users 
   WHERE username = USER;

5. Ver schemas/tablespaces acessíveis:
   SELECT DISTINCT owner 
   FROM all_tables 
   ORDER BY owner;

6. Ver permissões do usuário atual:
   SELECT * FROM user_sys_privs;
   SELECT * FROM user_role_privs;
   SELECT * FROM user_tab_privs;

7. Ver service names registrados:
   lsnrctl services
   # ou
   SELECT name, value FROM v$parameter WHERE name = 'service_names';

8. Ver conexões ativas:
   SELECT use

print(oracle_scot_cfg["url"])

In [53]:
print(oracle_scot_cfg["url"])

jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br


In [54]:
"""
Célula de diagnóstico para testar diferentes service names Oracle.
Execute esta célula no notebook para descobrir o service name correto.
"""
import os
from typing import Dict

from notebooks.test_oracle_service_names import test_oracle_service_names
oracle_scot_cfg = test_oracle_service_names("SCOT", tunnel=ssh_tunnel)

def test_oracle_service_names(prefix: str = "SCOT", tunnel=None, service_names: list = None):
    """Testa diferentes service names até encontrar um que funcione."""
    if service_names is None:
        service_names = [
            "bi.grupotracker.com.br",  # Original
            "ORCL",
            "ORCLPDB1",
            "XE",
            "ORCLCDB",
        ]
    
    print("=" * 70)
    print("TESTANDO DIFERENTES SERVICE NAMES")
    print("=" * 70)
    
    # Configura host e porta
    if tunnel and tunnel.is_active:
        host = "127.0.0.1"
        port = str(tunnel.local_bind_port)
        print(f"\n[INFO] Usando túnel SSH: {host}:{port}")
    else:
        host = os.getenv(f"ORACLE_{prefix}_HOST", "10.255.150.11")
        port = os.getenv(f"ORACLE_{prefix}_PORT", "1521")
        print(f"\n[INFO] Usando conexão direta: {host}:{port}")
    
    user = os.getenv(f"ORACLE_{prefix}_USER", "clickhouse")
    password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "qiU!E0oe")
    
    working_config = None
    
    for service in service_names:
        if not service:
            continue
            
        print(f"\n{'='*70}")
        print(f"[TESTE] Service Name: {service}")
        print('='*70)
        
        jdbc_url = f"jdbc:oracle:thin:@//{host}:{port}/{service}"
        print(f"URL: {jdbc_url}")
        
        cfg = {
            "url": jdbc_url,
            "properties": {
                "user": user,
                "password": password,
                "driver": "oracle.jdbc.driver.OracleDriver",
            },
        }
        
        try:
            # Testa com query simples (DUAL é uma tabela especial do Oracle)
            test_df = spark.read.jdbc(
                cfg["url"],
                table="(SELECT 'OK' as status, sysdate as data_hora FROM dual)",
                properties=cfg["properties"]
            )
            result = test_df.collect()
            print(f"\n[✓ SUCESSO] Service '{service}' funciona!")
            print(f"Resultado do teste: {result[0]}")
            working_config = cfg
            break
        except Exception as e:
            error_msg = str(e)
            if "ORA-12514" in error_msg:
                print(f"[✗ FALHOU] Service '{service}' não reconhecido pelo listener")
            elif "ORA-01017" in error_msg:
                print(f"[✗ FALHOU] Credenciais inválidas (usuário/senha)")
                print(f"          Verifique ORACLE_{prefix}_USER e ORACLE_{prefix}_PASSWORD")
            elif "ORA-12541" in error_msg or "TNS:no listener" in error_msg:
                print(f"[✗ FALHOU] Listener não está rodando ou não acessível")
            else:
                print(f"[✗ FALHOU] {error_msg[:200]}")
    
    if working_config:
        print(f"\n{'='*70}")
        print("[SUCESSO] Service name encontrado!")
        print('='*70)
        print(f"Service Name: {service}")
        print(f"URL JDBC: {working_config['url']}")
        print(f"\nConfigure no .env:")
        print(f"ORACLE_{prefix}_SERVICE={service}")
        return working_config
    else:
        print(f"\n{'='*70}")
        print("[ERRO] Nenhum service name funcionou!")
        print('='*70)
        print("\nPróximos passos:")
        print("1. Verifique o service name correto no servidor Oracle:")
        print("   ssh amaro.neto.beanalytic@10.255.150.11")
        print("   lsnrctl services")
        print("\n2. Ou verifique o SID:")
        print("   echo $ORACLE_SID")
        print("\n3. Se souber o SID, configure no .env:")
        print(f"   ORACLE_{prefix}_SID=SEU_SID")
        print("   E use formato SID na URL JDBC")
        return None

# Para usar no notebook:
# from notebooks.test_oracle_service_names import test_oracle_service_names
# oracle_scot_cfg = test_oracle_service_names("SCOT", tunnel=ssh_tunnel)


TESTANDO DIFERENTES SERVICE NAMES

[INFO] Usando túnel SSH: 127.0.0.1:53046

[TESTE] Service Name: bi.grupotracker.com.br
URL: jdbc:oracle:thin:@//127.0.0.1:53046/bi.grupotracker.com.br
[✗ FALHOU] name 'spark' is not defined

[TESTE] Service Name: ORCL
URL: jdbc:oracle:thin:@//127.0.0.1:53046/ORCL
[✗ FALHOU] name 'spark' is not defined

[TESTE] Service Name: ORCLPDB1
URL: jdbc:oracle:thin:@//127.0.0.1:53046/ORCLPDB1
[✗ FALHOU] name 'spark' is not defined

[TESTE] Service Name: XE
URL: jdbc:oracle:thin:@//127.0.0.1:53046/XE
[✗ FALHOU] name 'spark' is not defined

[TESTE] Service Name: ORCLCDB
URL: jdbc:oracle:thin:@//127.0.0.1:53046/ORCLCDB
[✗ FALHOU] name 'spark' is not defined

[ERRO] Nenhum service name funcionou!

Próximos passos:
1. Verifique o service name correto no servidor Oracle:
   ssh amaro.neto.beanalytic@10.255.150.11
   lsnrctl services

2. Ou verifique o SID:
   echo $ORACLE_SID

3. Se souber o SID, configure no .env:
   ORACLE_SCOT_SID=SEU_SID
   E use formato SID na UR

In [56]:
def list_oracle_tables(owner: str, cfg: Dict[str, Dict[str, str]], like: Optional[str] = None) -> DataFrame:
    """
    Lista tabelas do schema (owner) Oracle. Opcionalmente filtra por LIKE (maiúsculas).
    """
    owner_up = owner.upper()
    base_query = f"(SELECT table_name FROM all_tables WHERE owner = '{owner_up}'"
    if like:
        base_query += f" AND table_name LIKE '{like.upper()}'"
    base_query += ") tables_alias"

    return (
        spark.read.jdbc(cfg["url"], table=base_query, properties=cfg["properties"])
        .orderBy("table_name")
    )

list_oracle_tables("SCOT_OWNER", oracle_scot_cfg).show(20, truncate=False)

TypeError: 'NoneType' object is not subscriptable

In [ ]:
def profile_table(
        table: str,
        owner: str,
        cfg: Dict[str, Dict[str, str]],
        sample_rows: int = 20,
        stats_cols: Optional[list[str]] = None,
) -> None:
    """
    Lê uma tabela Oracle e exibe:
      - Schema
      - Contagem
      - Amostra
      - Estatísticas básicas de colunas numéricas (ou das fornecidas em stats_cols)
    """
    full_table = f"{owner}.{table}"
    df = spark.read.jdbc(cfg["url"], table=full_table, properties=cfg["properties"])

    print(f"== Schema: {full_table}")
    df.printSchema()

    print(f"\n== Contagem: {df.count():,}")
    print(f"\n== Amostra (até {sample_rows} linhas):")
    df.limit(sample_rows).show(truncate=False)

    numeric_cols = [f.name for f in df.schema.fields if
                    str(f.dataType).startswith("DecimalType") or str(f.dataType).startswith("LongType") or str(
                        f.dataType).startswith("IntegerType") or str(f.dataType).startswith("DoubleType") or str(
                        f.dataType).startswith("FloatType")]
    cols_to_stats = stats_cols if stats_cols else numeric_cols

    if cols_to_stats:
        print("\n== Estatísticas básicas:")
        df.select(
            *[
                F.expr(f"percentile_approx({c}, 0.5)").alias(f"{c}_p50")
                for c in cols_to_stats
            ],
            *[
                F.mean(c).alias(f"{c}_mean")
                for c in cols_to_stats
            ],
            *[
                F.stddev(c).alias(f"{c}_std")
                for c in cols_to_stats
            ],
            *[
                F.min(c).alias(f"{c}_min")
                for c in cols_to_stats
            ],
            *[
                F.max(c).alias(f"{c}_max")
                for c in cols_to_stats
            ],
        ).show(truncate=False)
    else:
        print("\n(Sem colunas numéricas para estatísticas.)")

# Exemplo: profile_table("MINHA_TABELA", "SCOT_OWNER", oracle_scot_cfg, sample_rows=10)
# Exemplo: profile_table("MINHA_TABELA", "GINF_OWNER", oracle_ginf_cfg, sample_rows=10, stats_cols=["COLUNA_NUM"])


In [ ]:
# ============================================================================
# GERENCIAMENTO DE TÚNEIS SSH
# ============================================================================

def list_all_ssh_tunnels():
    """Lista todos os túneis SSH ativos no notebook e no sistema."""
    import socket
    try:
        import psutil
        psutil_available = True
    except ImportError:
        psutil_available = False
    
    tunnels_found = []
    
    # 1. Verifica variáveis globais do notebook
    print("=" * 70)
    print("TÚNEIS SSH - VARIÁVEIS DO NOTEBOOK")
    print("=" * 70)
    
    try:
        if 'ssh_tunnel' in globals() and ssh_tunnel:
            tunnel_info = {
                'source': 'Variável ssh_tunnel',
                'active': ssh_tunnel.is_active if hasattr(ssh_tunnel, 'is_active') else False,
                'local_port': getattr(ssh_tunnel, 'local_bind_port', 'N/A'),
                'remote': f"{getattr(ssh_tunnel, 'remote_host', 'N/A')}:{getattr(ssh_tunnel, 'remote_port', 'N/A')}",
                'bastion': f"{getattr(ssh_tunnel, 'ssh_host', 'N/A')}:{getattr(ssh_tunnel, 'ssh_port', 'N/A')}",
                'tunnel_obj': ssh_tunnel
            }
            tunnels_found.append(tunnel_info)
            print(f"\n[1] Túnel encontrado em variável 'ssh_tunnel':")
            print(f"    Status: {'ATIVO' if tunnel_info['active'] else 'INATIVO'}")
            print(f"    Porta local: {tunnel_info['local_port']}")
            print(f"    Redireciona para: {tunnel_info['remote']}")
            print(f"    Bastion: {tunnel_info['bastion']}")
        else:
            print("\n[INFO] Nenhum túnel encontrado na variável 'ssh_tunnel'")
    except Exception as e:
        print(f"\n[ERRO] Erro ao verificar variável ssh_tunnel: {e}")
    
    # 2. Verifica processos sshtunnel
    print(f"\n{'=' * 70}")
    print("TÚNEIS SSH - PROCESSOS DO SISTEMA")
    print("=" * 70)
    
    if psutil_available:
        try:
            ssh_processes = []
            for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
                try:
                    if proc.info['name'] and 'ssh' in proc.info['name'].lower():
                        cmdline = ' '.join(proc.info['cmdline']) if proc.info['cmdline'] else ''
                        if 'sshtunnel' in cmdline or 'ssh -L' in cmdline or 'ssh -R' in cmdline:
                            ssh_processes.append({
                                'pid': proc.info['pid'],
                                'name': proc.info['name'],
                                'cmdline': cmdline[:100] if cmdline else 'N/A'
                            })
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    continue
            
            if ssh_processes:
            print(f"\n[2] Encontrados {len(ssh_processes)} processo(s) SSH relacionado(s) a túneis:")
            for i, proc in enumerate(ssh_processes, 1):
                print(f"    [{i}] PID: {proc['pid']}")
                print(f"        Comando: {proc['cmdline']}")
        else:
            print("\n[INFO] Nenhum processo SSH de túnel encontrado")
    except ImportError:
        print("\n[INFO] psutil não disponível. Instale com: pip install psutil")
    except Exception as e:
        print(f"\n[ERRO] Erro ao verificar processos: {e}")
    
    # 3. Verifica portas locais que podem ser túneis (range comum)
    print(f"\n{'=' * 70}")
    print("TÚNEIS SSH - PORTAS LOCAIS (50000-65000)")
    print("=" * 70)
    
    try:
        active_ports = []
        for port in range(50000, 65001, 1000):  # Verifica a cada 1000 portas para não demorar
            try:
                sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                sock.settimeout(0.1)
                result = sock.connect_ex(("127.0.0.1", port))
                sock.close()
                if result == 0:
                    active_ports.append(port)
            except:
                continue
        
        if active_ports:
            print(f"\n[3] Encontradas {len(active_ports)} porta(s) local(is) acessível(eis):")
            print(f"    Portas: {', '.join(map(str, active_ports[:20]))}")
            if len(active_ports) > 20:
                print(f"    ... e mais {len(active_ports) - 20} porta(s)")
            print(f"    [NOTA] Estas podem ser túneis SSH ou outros serviços")
        else:
            print("\n[INFO] Nenhuma porta suspeita encontrada no range comum")
    except Exception as e:
        print(f"\n[ERRO] Erro ao verificar portas: {e}")
    
    print(f"\n{'=' * 70}")
    print(f"TOTAL: {len(tunnels_found)} túnel(is) encontrado(s) em variáveis do notebook")
    print("=" * 70)
    
    return tunnels_found


def stop_all_ssh_tunnels():
    """Encerra todos os túneis SSH ativos."""
    try:
        import psutil
        psutil_available = True
    except ImportError:
        psutil_available = False
    
    stopped_count = 0
    
    print("=" * 70)
    print("ENCERRANDO TÚNEIS SSH")
    print("=" * 70)
    
    # 1. Encerra túneis em variáveis globais
    print("\n[1] Encerrando túneis em variáveis do notebook...")
    
    try:
        if 'ssh_tunnel' in globals() and ssh_tunnel:
            if hasattr(ssh_tunnel, 'is_active') and ssh_tunnel.is_active:
                if hasattr(ssh_tunnel, 'tunnel') and ssh_tunnel.tunnel:
                    try:
                        ssh_tunnel.tunnel.stop()
                        print(f"    ✓ Túnel na porta {getattr(ssh_tunnel, 'local_bind_port', 'N/A')} encerrado")
                        stopped_count += 1
                    except Exception as e:
                        print(f"    ✗ Erro ao encerrar túnel: {e}")
                else:
                    print(f"    ⚠ Túnel não tem objeto tunnel para encerrar")
            else:
                print(f"    ⚠ Túnel já está inativo")
        else:
            print(f"    [INFO] Nenhum túnel encontrado na variável 'ssh_tunnel'")
    except Exception as e:
        print(f"    [ERRO] Erro ao encerrar túnel: {e}")
    
    # 2. Encerra processos sshtunnel
    print(f"\n[2] Encerrando processos SSH de túneis...")
    
    if psutil_available:
        try:
            ssh_pids = []
            for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
            try:
                if proc.info['name'] and 'ssh' in proc.info['name'].lower():
                    cmdline = ' '.join(proc.info['cmdline']) if proc.info['cmdline'] else ''
                    if 'sshtunnel' in cmdline or ('ssh -L' in cmdline and '127.0.0.1' in cmdline):
                        ssh_pids.append(proc.info['pid'])
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        
        if ssh_pids:
            for pid in ssh_pids:
                try:
                    proc = psutil.Process(pid)
                    proc.terminate()
                    print(f"    ✓ Processo SSH PID {pid} encerrado")
                    stopped_count += 1
                except (psutil.NoSuchProcess, psutil.AccessDenied) as e:
                    print(f"    ✗ Não foi possível encerrar PID {pid}: {e}")
        else:
            print(f"    [INFO] Nenhum processo SSH de túnel encontrado")
        except Exception as e:
            print(f"    [ERRO] Erro ao encerrar processos: {e}")
    else:
        print(f"    [INFO] psutil não disponível. Apenas túneis em variáveis serão encerrados")
    
    # 3. Limpa variáveis globais
    print(f"\n[3] Limpando variáveis globais...")
    try:
        if 'ssh_tunnel' in globals():
            globals()['ssh_tunnel'] = None
            print(f"    ✓ Variável 'ssh_tunnel' limpa")
        if 'ssh_tunnel_manager' in globals():
            globals()['ssh_tunnel_manager'] = None
            print(f"    ✓ Variável 'ssh_tunnel_manager' limpa")
    except Exception as e:
        print(f"    [ERRO] Erro ao limpar variáveis: {e}")
    
    print(f"\n{'=' * 70}")
    print(f"TOTAL: {stopped_count} túnel(is) encerrado(s)")
    print("=" * 70)
    
    return stopped_count


# Executa listagem
print("\n")
tunnels = list_all_ssh_tunnels()


IndentationError: expected an indented block after 'if' statement on line 63 (2926780670.py, line 64)

# ============================================================================
# STATUS DO USUÁRIO C##AMARO_BE
# ============================================================================

## Status atual:
- **Usuário**: C##AMARO_BE
- **Status**: OPEN ✅
- **Criado em**: 12-DEC-25
- **Tablespace padrão**: USERS

## Próximo passo:
Como o usuário já existe e está ativo, apenas redefina a senha:

```sql
ALTER USER C##AMARO_BE IDENTIFIED BY "qiU!E0oe123";
ALTER USER C##AMARO_BE ACCOUNT UNLOCK;
```

## Depois de redefinir a senha:
Atualize o arquivo `.env`:
```bash
ORACLE_SCOT_USER=C##AMARO_BE
ORACLE_SCOT_PASSWORD=qiU!E0oe123
```


# ============================================================================
# EXPLORAR SCHEMA GINF
# ============================================================================

## Schema a explorar: GINF

Vamos começar listando as tabelas disponíveis no schema GINF.

In [ ]:
# ============================================================================
# LISTAR TABELAS DO SCHEMA GINF
# ============================================================================

def list_ginf_tables(cfg, like_filter=None):
    """
    Lista todas as tabelas do schema GINF.
    
    Args:
        cfg: Configuração Oracle (oracle_ginf_cfg)
        like_filter: Filtro opcional para nome da tabela (ex: 'VENDA%')
    """
    # Verifica se Spark está disponível
    if 'spark' not in globals():
        print("[ERRO] Spark não está disponível!")
        print("[INFO] Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        return None
    
    print("=" * 70)
    print("TABELAS DO SCHEMA GINF")
    print("=" * 70)
    
    try:
        base_query = """
            SELECT 
                owner,
                table_name,
                tablespace_name,
                num_rows,
                last_analyzed
            FROM all_tables
            WHERE owner = 'GINF'
        """
        
        if like_filter:
            base_query += f" AND table_name LIKE '{like_filter.upper()}'"
        
        base_query += " ORDER BY table_name"
        
        df = spark.read.jdbc(
            cfg["url"],
            table=f"({base_query}) tables_alias",
            properties=cfg["properties"]
        )
        
        print(f"\n[INFO] Encontradas {df.count()} tabela(s) no schema GINF")
        print("\n" + "=" * 70)
        df.show(truncate=False)
        print("=" * 70)
        
        return df
        
    except Exception as e:
        error_msg = str(e)
        print(f"\n✗ ERRO ao listar tabelas: {error_msg[:300]}")
        
        if "name 'spark' is not defined" in error_msg:
            print("\n[ERRO] Spark não está definido")
            print("  Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        elif "ORA-01017" in error_msg:
            print("\n[ERRO] Credenciais inválidas")
            print("  Verifique usuário e senha")
        elif "ORA-12514" in error_msg:
            print("\n[ERRO] Service name não reconhecido")
            print("  Verifique o service name na URL")
        elif "ORA-00942" in error_msg:
            print("\n[ERRO] Tabela ou view não existe")
            print("  O usuário pode não ter permissão para acessar all_tables")
        
        return None


# Lista todas as tabelas do GINF
if 'oracle_ginf_cfg' in globals() and 'spark' in globals():
    print("\n")
    ginf_tables = list_ginf_tables(oracle_ginf_cfg)
else:
    if 'oracle_ginf_cfg' not in globals():
        print("[AVISO] Execute primeiro a célula de configuração Oracle (célula 7)")
    if 'spark' not in globals():
        print("[AVISO] Execute primeiro a célula 3 (%run LoadEnvAndSetupSession.py) para inicializar o Spark")




TABELAS DO SCHEMA GINF

✗ ERRO ao listar tabelas: An error occurred while calling o86.jdbc.
: java.sql.SQLRecoverableException: Erro de ES: The Network Adapter could not establish the connection (CONNECTION_ID=NEbuA4JaSR+dLBQEEifynA==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CCon


In [ ]:
# ============================================================================
# TESTAR CONEXÃO COM SCHEMA GINF
# ============================================================================

def test_ginf_connection(cfg):
    """
    Testa a conexão com o schema GINF executando uma query simples.
    """
    # Verifica se Spark está disponível
    if 'spark' not in globals():
        print("[ERRO] Spark não está disponível!")
        print("[INFO] Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        return False
    
    print("=" * 70)
    print("TESTE DE CONEXÃO - SCHEMA GINF")
    print("=" * 70)
    
    try:
        # Query simples para testar conexão
        test_query = """
            SELECT 
                USER as current_user,
                SYS_CONTEXT('USERENV', 'SESSION_USER') as session_user,
                SYSDATE as current_time,
                COUNT(*) as table_count
            FROM all_tables
            WHERE owner = 'GINF'
        """
        
        df = spark.read.jdbc(
            cfg["url"],
            table=f"({test_query}) test_alias",
            properties=cfg["properties"]
        )
        
        result = df.collect()[0]
        
        print(f"\n✓ Conexão estabelecida com sucesso!")
        print(f"  Usuário: {result['current_user']}")
        print(f"  Sessão: {result['session_user']}")
        print(f"  Hora do servidor: {result['current_time']}")
        print(f"  Tabelas no schema GINF: {result['table_count']}")
        print("=" * 70)
        
        return True
        
    except Exception as e:
        error_msg = str(e)
        print(f"\n✗ ERRO na conexão: {error_msg[:300]}")
        
        if "name 'spark' is not defined" in error_msg:
            print("\n[ERRO] Spark não está definido")
            print("  Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        elif "ORA-01017" in error_msg:
            print("\n[ERRO] Credenciais inválidas")
            print("  Verifique usuário e senha")
        elif "ORA-12514" in error_msg:
            print("\n[ERRO] Service name não reconhecido")
            print("  Verifique o service name na URL")
        elif "ORA-00942" in error_msg:
            print("\n[ERRO] Sem permissão para acessar all_tables")
            print("  O usuário pode precisar de permissões adicionais")
        
        return False


# Testa conexão
if 'oracle_ginf_cfg' in globals() and 'spark' in globals():
    print("\n")
    test_ginf_connection(oracle_ginf_cfg)
else:
    if 'oracle_ginf_cfg' not in globals():
        print("[AVISO] Execute primeiro a célula de configuração Oracle (célula 7)")
    if 'spark' not in globals():
        print("[AVISO] Execute primeiro a célula 3 (%run LoadEnvAndSetupSession.py) para inicializar o Spark")


In [ ]:
# ============================================================================
# EXPLORAR ESTRUTURA DE UMA TABELA GINF
# ============================================================================

def explore_ginf_table(cfg, table_name, sample_rows=10):
    """
    Explora uma tabela específica do schema GINF.
    
    Args:
        cfg: Configuração Oracle (oracle_ginf_cfg)
        table_name: Nome da tabela (sem o schema, ex: 'VENDA')
        sample_rows: Número de linhas para amostra
    """
    # Verifica se Spark está disponível
    if 'spark' not in globals():
        print("[ERRO] Spark não está disponível!")
        print("[INFO] Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        return None
    
    print("=" * 70)
    print(f"EXPLORANDO TABELA: GINF.{table_name.upper()}")
    print("=" * 70)
    
    try:
        # 1. Estrutura da tabela (colunas)
        print(f"\n[1] Estrutura da tabela:")
        columns_query = f"""
            SELECT 
                column_name,
                data_type,
                data_length,
                data_precision,
                data_scale,
                nullable
            FROM all_tab_columns
            WHERE owner = 'GINF' AND table_name = '{table_name.upper()}'
            ORDER BY column_id
        """
        
        df_columns = spark.read.jdbc(
            cfg["url"],
            table=f"({columns_query}) columns_alias",
            properties=cfg["properties"]
        )
        df_columns.show(truncate=False)
        
        # 2. Contagem de linhas
        print(f"\n[2] Contagem de linhas:")
        count_query = f"SELECT COUNT(*) as total_rows FROM GINF.{table_name.upper()}"
        df_count = spark.read.jdbc(
            cfg["url"],
            table=f"({count_query}) count_alias",
            properties=cfg["properties"]
        )
        total_rows = df_count.collect()[0]['total_rows']
        print(f"  Total de linhas: {total_rows:,}")
        
        # 3. Amostra de dados
        if total_rows > 0:
            print(f"\n[3] Amostra de dados (primeiras {sample_rows} linhas):")
            sample_query = f"SELECT * FROM GINF.{table_name.upper()} WHERE ROWNUM <= {sample_rows}"
            df_sample = spark.read.jdbc(
                cfg["url"],
                table=f"({sample_query}) sample_alias",
                properties=cfg["properties"]
            )
            df_sample.show(truncate=False)
        else:
            print(f"\n[3] Tabela vazia (sem dados)")
        
        print("=" * 70)
        
        return df_columns
        
    except Exception as e:
        error_msg = str(e)
        print(f"\n✗ ERRO ao explorar tabela: {error_msg[:300]}")
        
        if "name 'spark' is not defined" in error_msg:
            print("\n[ERRO] Spark não está definido")
            print("  Execute a célula 3 (%run LoadEnvAndSetupSession.py) primeiro")
        elif "ORA-00942" in error_msg:
            print(f"\n[ERRO] Tabela GINF.{table_name.upper()} não existe")
            print("  Verifique o nome da tabela")
        elif "ORA-01031" in error_msg:
            print("\n[ERRO] Sem permissão para acessar esta tabela")
        
        return None


# Exemplo de uso:
# explore_ginf_table(oracle_ginf_cfg, "VENDA", sample_rows=10)
# explore_ginf_table(oracle_ginf_cfg, "PRODUTO", sample_rows=5)

print("""
Para explorar uma tabela específica do GINF, use:

    explore_ginf_table(oracle_ginf_cfg, "NOME_DA_TABELA", sample_rows=10)

Exemplo:
    explore_ginf_table(oracle_ginf_cfg, "VENDA", sample_rows=10)
""")


In [ ]:
# ============================================================================
# DIAGNÓSTICO DO SPARK
# ============================================================================

def diagnose_spark():
    """Diagnostica o estado do Spark no notebook."""
    print("=" * 70)
    print("DIAGNÓSTICO DO SPARK")
    print("=" * 70)
    
    # 1. Verifica se spark está em globals
    if 'spark' in globals():
        print("\n[1] ✓ Variável 'spark' encontrada em globals()")
        try:
            # Testa se o Spark está funcionando
            test_df = spark.range(1).limit(1)
            result = test_df.collect()
            print(f"    ✓ Spark está funcionando")
            print(f"    ✓ Versão: {spark.version}")
            
            # Informações da sessão
            print(f"\n[2] Informações da sessão:")
            print(f"    App Name: {spark.sparkContext.appName}")
            print(f"    Master: {spark.sparkContext.master}")
            
            # UI Web URL
            try:
                ui_url = spark.sparkContext.uiWebUrl
                if ui_url:
                    print(f"    Spark UI: {ui_url}")
            except:
                print(f"    Spark UI: N/A")
            
            # Verifica se tem driver JDBC Oracle
            print(f"\n[3] Verificando driver JDBC Oracle...")
            try:
                jars = spark.sparkContext.getConf().get("spark.jars.packages", "")
                if "ojdbc" in jars.lower() or "oracle" in jars.lower():
                    print(f"    ✓ Driver Oracle configurado: {jars}")
                else:
                    print(f"    ⚠ Driver Oracle pode não estar configurado")
                    print(f"    Jars configurados: {jars if jars else 'Nenhum'}")
            except:
                print(f"    ⚠ Não foi possível verificar jars")
            
            print("=" * 70)
            return True
            
        except Exception as e:
            print(f"    ✗ Erro ao testar Spark: {e}")
            print("=" * 70)
            return False
    else:
        print("\n[1] ✗ Variável 'spark' NÃO encontrada em globals()")
        print("\n[SOLUÇÃO] Execute a célula 3:")
        print("    %run LoadEnvAndSetupSession.py")
        print("=" * 70)
        return False


# Executa diagnóstico
diagnose_spark()


In [ ]:
# ============================================================================
# ENCERRAR TODOS OS TÚNEIS SSH
# ============================================================================

# Para encerrar todos os túneis SSH ativos, execute a função abaixo:
# stop_all_ssh_tunnels()

# Ou execute diretamente (descomente a linha abaixo):
# stop_all_ssh_tunnels()


In [ ]:
display(df)